In [ ]:
import numpy as np
import skimage as sk
import os
from glob import glob
import re
import random
import seaborn_image as isns

In [ ]:
# get all images that have paired detection overlays
vai_files = sorted(glob('/home/gkristin/Desktop/Trick_vial_manus/Figures/Raw_Images_for_Figs/VAI_Images/*.png'))

In [ ]:
a6 = []
b5 = []
c1 = []
c4 = []
d3 = []

for file in vai_files:
    if 'A6' in os.path.basename(file):
        a6.append(file)
    elif 'B5' in os.path.basename(file):
        b5.append(file)
    elif 'C1' in os.path.basename(file):
        c1.append(file)
    elif 'C4' in os.path.basename(file):
        c4.append(file)
    elif 'D3' in os.path.basename(file):
        d3.append(file)
    else:
        print(f'{os.path.basename(file)} not matching pattern search, skipping')


In [ ]:
len(c1)

In [ ]:
def sort_imgs_detections(file_list):
    detections = []
    imgs = []
    for file in file_list:
        if 'Detections' in os.path.basename(file):
            detections.append(file)
            print(f'Added {file} to detection list')
        else:
            imgs.append(file)
            print(f'Added {file} to image list')
    return detections, imgs

def random_subset(detections, imgs, n):
    random_selection_detections = random.sample(detections,n)
    print(f'finding matches for: {list(map(os.path.basename,random_selection_detections))}')
    random_selection_imgs = []
    for sample in random_selection_detections:
        name = re.search(r'[0-9]{6}_[A-z][0-9]', os.path.basename(sample)).group()
        print(f'looking for {name}')
        img_match = next(img for img in imgs if name in os.path.basename(img))
        random_selection_imgs.append(img_match)
        print(f'found {name} and added to list')
    return random_selection_imgs, random_selection_detections


def pair_imgs_detections(random_selection_imgs, random_selection_detections):
    paired_imgs = []
    tma_loc = re.search(r'[A-Z][1-9]',random_selection_imgs[0]).group()
    print(f'creating image pairs for {tma_loc}')
    print(tma_loc)
    for img, detection in zip(random_selection_imgs, random_selection_detections):
        pair = [img, detection]
        paired_stack = list(map(sk.io.imread,pair))
        moved_ch_axis = [np.moveaxis(arr,-1,0) for arr in paired_stack]
        stacked_arr = np.stack(moved_ch_axis,axis=0)
        paired_imgs.append(stacked_arr)
    return paired_imgs, tma_loc

def stack_and_save(paired_imgs, path, tma_id):
    stacked_pairs = np.stack(paired_imgs,axis=0)
    sk.io.imsave(os.path.join(path,f'{tma_id}_concatenated_pairs.tif'),stacked_pairs)
    print(f'saved stack for {tma_id}')

def process(list_of_files, path):
    for list in list_of_files:
        detections, imgs = sort_imgs_detections(list)
        random_selection_imgs, random_selection_detections = random_subset(detections, imgs, 3)
        paired_imgs, tma_id = pair_imgs_detections(random_selection_imgs, random_selection_detections)
        save_loc = os.path.join(path,f'{tma_id}_paired')
        os.makedirs(save_loc,exist_ok=True)
        stack_and_save(paired_imgs, save_loc, tma_id)

In [ ]:
process([a6,b5,c1,c4,d3], path='/home/gkristin/Desktop/Trick_vial_manus/Figures/Raw_Images_for_Figs/VAI_Images/')

In [ ]:
paired_imgs = []

for img in imgs:
    tma_id = os.path.basename(img)[:-4]
    detection_match = next(detection for detection in detections if tma_id in os.path.basename(detection))
    pair = [img, detection_match]
    paired_stack = list(map(sk.io.imread,pair))
    moved_ch_axis = [np.moveaxis(arr,-1,0) for arr in paired_stack]
    stacked_arr = np.stack(moved_ch_axis,axis=0)
    paired_imgs.append(stacked_arr)

    

In [ ]:
stacked_pairs = np.stack(paired_imgs,axis=0)

In [ ]:
stacked_pairs.shape

In [ ]:
moved_ch_axis = np.moveaxis()

In [ ]:
import napari

In [ ]:
viewer = napari.Viewer()
viewer.add_image(stacked_pairs)

In [ ]:
path = '/home/gkristin/Desktop/Trick_vial_manus/Figures/Raw_Images_for_Figs/VAI_Images/'
sk.io.imsave(os.path.join(path,'concatenated_pairs.tif'),stacked_pairs)

In [ ]:
%reset